# Detecting patterns of speciation in the fossil record

This notebook solves exercises 1–10 of the fossil-analysis project for
**Data Analysis with Python 2024–2025**. It follows the
[official assignment](https://github.com/rage/data-analysis-with-python-project-assignments/blob/main/fossil-analysis-assignment-2020.pdf)
and uses the public NOW export preserved with a completed course project.

The analysis cleans the fossil occurrences, assigns MN/MQ time units, finds first
occurrences, measures local sampling density, fits a logistic regression model,
and flags locality/time combinations with unusually many first occurrences.

## Exercises 1–2 — load the NOW data

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import binom

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "thefl0ur/fossil_data_analysis/master/raw_data.txt"
)
DATA_FILE = Path("raw_data.txt")

if not DATA_FILE.exists():
    urlretrieve(DATA_URL, DATA_FILE)

raw_fossils = pd.read_csv(DATA_FILE)
print(f"Raw data: {raw_fossils.shape[0]:,} rows and {raw_fossils.shape[1]} columns")
raw_fossils.head()

## Exercise 3 — clean and preprocess the occurrences

In [ ]:
TIME_BOUNDS = [
    ("MN1", 23.0, 21.7),
    ("MN2", 21.7, 19.5),
    ("MN3", 19.5, 17.2),
    ("MN4", 17.2, 16.4),
    ("MN5", 16.4, 14.2),
    ("MN6", 14.2, 12.85),
    ("MN7-8", 12.85, 11.2),
    ("MN9", 11.2, 9.9),
    ("MN10", 9.9, 8.9),
    ("MN11", 8.9, 7.6),
    ("MN12", 7.6, 7.1),
    ("MN13", 7.1, 5.3),
    ("MN14", 5.3, 5.0),
    ("MN15", 5.0, 3.55),
    ("MN16", 3.55, 2.5),
    ("MN17", 2.5, 1.9),
    ("MQ18", 1.9, 0.85),
    ("MQ19", 0.85, 0.01),
]
TIME_UNITS = ["pre-MN"] + [name for name, _, _ in TIME_BOUNDS] + ["post-MN"]
TIME_ORDER = {name: index for index, name in enumerate(TIME_UNITS)}


def assign_time_unit(mean_age):
    if mean_age > TIME_BOUNDS[0][1]:
        return "pre-MN"
    for name, upper, lower in TIME_BOUNDS:
        # Iterating from old to young assigns an exact boundary to the older unit.
        if lower <= mean_age <= upper:
            return name
    return "post-MN"


columns = [
    "LIDNUM", "NAME", "LAT", "LONG", "MAX_AGE", "MIN_AGE", "GENUS", "SPECIES"
]
fossils = raw_fossils[columns].copy()
fossils = fossils.loc[
    ~((fossils["LAT"] == 0) & (fossils["LONG"] == 0))
    & ~fossils["SPECIES"].isin(["sp.", "indet."])
].copy()

mean_age = (fossils["MIN_AGE"] + fossils["MAX_AGE"]) / 2
fossils["MN_UNIT"] = mean_age.map(assign_time_unit)

# The current public export writes the Roman numeral I as the digit 1.
fossils.loc[fossils["NAME"] == "Samos Main Bone Beds", "MN_UNIT"] = "MN12"
fossils.loc[
    fossils["NAME"].isin(["Can Llobateres I", "Can Llobateres 1"]), "MN_UNIT"
] = "MN9"

fossils["SPECIES_ID"] = fossils.groupby(["GENUS", "SPECIES"]).ngroup() + 1
fossils = fossils.drop_duplicates(["LIDNUM", "SPECIES_ID"]).copy()
fossils["TIME_ORDER"] = fossils["MN_UNIT"].map(TIME_ORDER)

print(f"Clean data: {len(fossils):,} rows")
print(f"Unique species: {fossils['SPECIES_ID'].nunique():,}")
print(f"Unique localities: {fossils['LIDNUM'].nunique():,}")
fossils.head()

## Exercise 4 — first occurrences through time

In [ ]:
oldest_order = fossils.groupby("SPECIES_ID")["TIME_ORDER"].transform("min")
fossils["IS_FIRST_OCC"] = fossils["TIME_ORDER"].eq(oldest_order)

occurrences_by_species = (
    fossils.groupby(["SPECIES_ID", "MN_UNIT"], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=TIME_UNITS, fill_value=0)
)

first_unit_by_species = (
    fossils.loc[fossils["IS_FIRST_OCC"], ["SPECIES_ID", "MN_UNIT", "TIME_ORDER"]]
    .sort_values("TIME_ORDER")
    .drop_duplicates("SPECIES_ID")
    .set_index("SPECIES_ID")[["MN_UNIT"]]
)

time_summary = (
    fossils.groupby(["MN_UNIT", "TIME_ORDER"], observed=True)
    .agg(
        TOTAL_OCCURRENCES=("SPECIES_ID", "size"),
        FIRST_OCCURRENCES=("IS_FIRST_OCC", "sum"),
    )
    .reset_index()
    .sort_values("TIME_ORDER")
)
time_summary["FIRST_PROPORTION"] = (
    time_summary["FIRST_OCCURRENCES"] / time_summary["TOTAL_OCCURRENCES"]
)

fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
axes[0].plot(time_summary["MN_UNIT"], time_summary["FIRST_PROPORTION"], marker="o")
axes[0].set_ylabel("Proportion of first occurrences")
axes[0].grid(alpha=0.25)
axes[1].plot(time_summary["MN_UNIT"], time_summary["TOTAL_OCCURRENCES"], marker="o")
axes[1].set_ylabel("Total occurrences")
axes[1].set_xlabel("Time unit")
axes[1].grid(alpha=0.25)
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()

time_summary

## Exercise 5 — locality statistics and sampling patterns

In [ ]:
localities = (
    fossils.groupby("LIDNUM", as_index=False)
    .agg(
        NAME=("NAME", "first"),
        LAT=("LAT", "first"),
        LONG=("LONG", "first"),
        MN_UNIT=("MN_UNIT", "first"),
        TIME_ORDER=("TIME_ORDER", "first"),
        FIRST_OCCURRENCES=("IS_FIRST_OCC", "sum"),
        TOTAL_OCCURRENCES=("SPECIES_ID", "size"),
    )
)
localities["FIRST_PROPORTION"] = (
    localities["FIRST_OCCURRENCES"] / localities["TOTAL_OCCURRENCES"]
)


def sampling_region(frame):
    conditions = [
        frame["LAT"].gt(35) & frame["LONG"].between(-25, 40),
        frame["LAT"].between(-35, 35) & frame["LONG"].between(-20, 55),
        frame["LAT"].gt(0) & frame["LONG"].between(40, 180),
    ]
    return np.select(conditions, ["Europe", "Africa", "Asia"], default="Other")


localities["REGION"] = sampling_region(localities)
regional_sampling = (
    localities.loc[localities["REGION"].isin(["Europe", "Africa", "Asia"])]
    .groupby("REGION")
    .agg(
        LOCALITIES=("LIDNUM", "nunique"),
        TOTAL_OCCURRENCES=("TOTAL_OCCURRENCES", "sum"),
        MEAN_OCCURRENCES_PER_LOCALITY=("TOTAL_OCCURRENCES", "mean"),
    )
)


def load_world_map():
    try:
        import geopandas as gpd
        try:
            path = gpd.datasets.get_path("naturalearth_lowres")
        except (AttributeError, ImportError):
            from geodatasets import get_path
            path = get_path("naturalearth.land")
        return gpd.read_file(path)
    except (ImportError, OSError):
        return None


world = load_world_map()
fig, axes = plt.subplots(6, 4, figsize=(18, 22))
for ax, unit in zip(axes.flat, TIME_UNITS):
    if world is not None:
        world.plot(ax=ax, color="wheat", edgecolor="0.4", linewidth=0.3)
    points = localities.loc[localities["MN_UNIT"] == unit]
    sizes = 8 + 90 * points["TOTAL_OCCURRENCES"] / max(
        points["TOTAL_OCCURRENCES"].max(), 1
    )
    ax.scatter(points["LONG"], points["LAT"], s=sizes, alpha=0.55)
    ax.set_title(unit)
    ax.set_xlim(-180, 180)
    ax.set_ylim(-60, 90)
for ax in axes.flat[len(TIME_UNITS):]:
    ax.set_visible(False)
plt.tight_layout()

regional_sampling

## Exercise 6 — sampling in each focal 10° × 10° area

In [ ]:
previous_unit = {
    unit: (TIME_UNITS[index - 1] if index else None)
    for index, unit in enumerate(TIME_UNITS)
}

unit_arrays = {}
for unit, group in fossils.groupby("MN_UNIT"):
    unit_arrays[unit] = {
        "lat": group["LAT"].to_numpy(),
        "long": group["LONG"].to_numpy(),
        "first": group["IS_FIRST_OCC"].to_numpy(dtype=int),
    }


def counts_in_square(unit, latitude, longitude):
    values = unit_arrays.get(unit)
    if values is None:
        return 0, 0
    inside = (
        (np.abs(values["lat"] - latitude) <= 5)
        & (np.abs(values["long"] - longitude) <= 5)
    )
    return int(inside.sum()), int(values["first"][inside].sum())


area_statistics = []
for locality in localities.itertuples(index=False):
    current_total, current_first = counts_in_square(
        locality.MN_UNIT, locality.LAT, locality.LONG
    )
    preceding_total, _ = counts_in_square(
        previous_unit[locality.MN_UNIT], locality.LAT, locality.LONG
    )
    area_statistics.append((current_total, current_first, preceding_total))

localities[
    ["CURRENT_AREA_OCC", "FIRST_AREA_OCC", "PREVIOUS_AREA_OCC"]
] = pd.DataFrame(area_statistics, index=localities.index)

localities.head()

## Exercise 7 — logistic regression for European localities

In [ ]:
occurrences_with_sampling = fossils.merge(
    localities[["LIDNUM", "PREVIOUS_AREA_OCC"]], on="LIDNUM", how="left"
)
europe_occurrences = occurrences_with_sampling.loc[
    occurrences_with_sampling["LONG"].between(-25, 40, inclusive="neither")
    & occurrences_with_sampling["LAT"].gt(35)
    & occurrences_with_sampling["TIME_ORDER"].between(
        TIME_ORDER["MN2"], TIME_ORDER["MQ19"]
    )
].copy()

regression_data = europe_occurrences[
    ["PREVIOUS_AREA_OCC", "IS_FIRST_OCC"]
].astype({"PREVIOUS_AREA_OCC": float, "IS_FIRST_OCC": int})

X = sm.add_constant(regression_data[["PREVIOUS_AREA_OCC"]], has_constant="add")
y = regression_data["IS_FIRST_OCC"]
logit_result = sm.Logit(y, X).fit(disp=False)
print(logit_result.summary())

x_grid = np.linspace(0, regression_data["PREVIOUS_AREA_OCC"].max(), 250)
X_grid = pd.DataFrame({"const": 1.0, "PREVIOUS_AREA_OCC": x_grid})
prediction = logit_result.get_prediction(X_grid).summary_frame(alpha=0.05)

predicted_column = "predicted" if "predicted" in prediction else "mean"
lower_column = "ci_lower" if "ci_lower" in prediction else "mean_ci_lower"
upper_column = "ci_upper" if "ci_upper" in prediction else "mean_ci_upper"

plt.figure(figsize=(9, 5))
plt.plot(x_grid, prediction[predicted_column], color="tab:red", label="Logistic fit")
plt.fill_between(
    x_grid,
    prediction[lower_column],
    prediction[upper_column],
    color="tab:red",
    alpha=0.2,
    label="95% confidence interval",
)
plt.xlabel("Occurrences in the focal area during the preceding time unit")
plt.ylabel("Probability of a first occurrence")
plt.legend()
plt.tight_layout()

## Exercises 8–9 — expected proportions and significance

In [ ]:
europe_localities = localities.loc[
    localities["LONG"].between(-25, 40, inclusive="neither")
    & localities["LAT"].gt(35)
    & localities["TIME_ORDER"].between(TIME_ORDER["MN2"], TIME_ORDER["MQ19"])
].copy()

locality_predictors = pd.DataFrame(
    {
        "const": 1.0,
        "PREVIOUS_AREA_OCC": europe_localities["PREVIOUS_AREA_OCC"].astype(float),
    },
    index=europe_localities.index,
)
europe_localities["EXPECTED_FIRST_PROPORTION"] = logit_result.predict(
    locality_predictors
)
europe_localities["P_VALUE"] = binom.sf(
    europe_localities["FIRST_AREA_OCC"] - 1,
    europe_localities["CURRENT_AREA_OCC"],
    europe_localities["EXPECTED_FIRST_PROPORTION"],
)

significant_localities = europe_localities.loc[
    europe_localities["P_VALUE"] <= 0.05
].sort_values(["TIME_ORDER", "P_VALUE"])

print("Significant locality/time combinations:", len(significant_localities))
print(
    significant_localities.groupby("MN_UNIT", observed=True)
    .size()
    .rename("SIGNIFICANT_LOCALITIES")
)
significant_localities[
    [
        "LIDNUM", "NAME", "MN_UNIT", "FIRST_AREA_OCC", "CURRENT_AREA_OCC",
        "EXPECTED_FIRST_PROPORTION", "P_VALUE",
    ]
].head(20)

## Exercise 10 — map statistically significant localities

In [ ]:
plot_units = TIME_UNITS[TIME_ORDER["MN2"]:TIME_ORDER["MQ19"] + 1]
fig, axes = plt.subplots(6, 3, figsize=(16, 27))
color_norm = plt.Normalize(0, 0.05)
color_map = plt.colormaps["viridis_r"]

for ax, unit in zip(axes.flat, plot_units):
    if world is not None:
        world.plot(ax=ax, color="wheat", edgecolor="0.45", linewidth=0.4)
    points = europe_localities.loc[europe_localities["MN_UNIT"] == unit]
    significant = points["P_VALUE"] <= 0.05
    ax.scatter(
        points.loc[~significant, "LONG"],
        points.loc[~significant, "LAT"],
        s=14,
        color="0.65",
        alpha=0.45,
        label="p > 0.05",
    )
    marked = ax.scatter(
        points.loc[significant, "LONG"],
        points.loc[significant, "LAT"],
        c=points.loc[significant, "P_VALUE"],
        cmap=color_map,
        norm=color_norm,
        s=48,
        edgecolor="black",
        linewidth=0.5,
        label="p ≤ 0.05",
    )
    ax.set_title(unit)
    ax.set_xlim(-25, 40)
    ax.set_ylim(35, 72)
    ax.grid(alpha=0.15)

for ax in axes.flat[len(plot_units):]:
    ax.set_visible(False)

fig.colorbar(marked, ax=axes, shrink=0.35, label="p-value")
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2)
plt.show()

## Interpretation

Sampling density varies strongly over both space and time, so raw counts of first
occurrences are not sufficient evidence for a “species factory”. The logistic
model estimates the baseline probability of a first occurrence from sampling in
the preceding time unit. The binomial survival probability then identifies
locality/time combinations whose observed first-occurrence count is unusually
large relative to that baseline. The table and maps above provide the exact
candidate localities; points with `p ≤ 0.05` are the statistically significant
ones.